# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/usr/bin/python3


In [5]:
# The notebook's instructions advise against installing/uninstalling TensorFlow packages.
# This cell was causing a numpy version conflict, so its content has been removed.
# Please restart the Colab runtime after this change.

In [6]:
# The notebook's instructions advise against installing/uninstalling TensorFlow packages.
# This cell was causing a Keras/TensorFlow version conflict, so its content has been removed.
# Please restart the Colab runtime after this change.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

AttributeError: module 'tensorflow._api.v2.compat.v2.__internal__' has no attribute 'register_load_context_function'


---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [3]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [4]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
# Step 1: Separate the feature matrix and class labels.
X = df[feature_names].values
y = df["Class"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique classes:", np.unique(y))

X shape: (178, 13)
y shape: (178,)
Unique classes: [0 1 2]


In [5]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)

X_train shape: (124, 13)
X_test shape:  (54, 13)
y_train shape: (124,)
y_test shape:  (54,)


In [ ]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#

In [ ]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#


In [8]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
# Step 5: Define a Sequential model
# Step 5: Define a Sequential model
# Re-define aliases (since the original import cell crashed)
Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense

# Step 5: Define a Sequential model
model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [10]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
# Step 6: Compile the model
# Re-define missing alias
to_categorical = tf.keras.utils.to_categorical

# Step 6: Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Convert integer labels to one-hot encoded vectors
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat  = to_categorical(y_test,  num_classes=num_classes)

# Train the model
history = model.fit(
    X_train, y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 2s 21ms/step - loss: 49.5872 - accuracy: 0.3232 - val_loss: 21.9180 - val_accuracy: 0.4400
Epoch 2/20
13/13 [==============================] - 0s 7ms/step - loss: 14.1749 - accuracy: 0.4343 - val_loss: 1.0411 - val_accuracy: 0.5200
Epoch 3/20
13/13 [==============================] - 0s 6ms/step - loss: 4.9619 - accuracy: 0.4646 - val_loss: 5.1588 - val_accuracy: 0.6000
Epoch 4/20
13/13 [==============================] - 0s 6ms/step - loss: 2.7931 - accuracy: 0.6061 - val_loss: 1.3787 - val_accuracy: 0.4400
Epoch 5/20
13/13 [==============================] - 0s 6ms/step - loss: 0.8451 - accuracy: 0.7273 - val_loss: 0.9632 - val_accuracy: 0.7200
Epoch 6/20
13/13 [==============================] - 0s 6ms/step - loss: 0.8053 - accuracy: 0.6869 - val_loss: 0.8452 - val_accuracy: 0.7600
Epoch 7/20
13/13 [==============================] - 0s 6ms/step - loss: 0.8252 - accuracy: 0.6566 - val_loss: 1.7896 - val_accuracy: 0.5200
Epoch 8/20
13/13

In [11]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
# Step 7: Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Convert one-hot predictions back to integer class labels
y_pred_cat = model.predict(X_test)
y_pred     = np.argmax(y_pred_cat, axis=1)
y_true     = np.argmax(y_test_cat, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=wine.target_names))

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Test Loss:     0.2883
Test Accuracy: 0.9074
2/2 [==============================] - 0s 13ms/step

Classification Report:
              precision    recall  f1-score   support

     class_0       1.00      0.89      0.94        19
     class_1       0.81      1.00      0.89        21
     class_2       1.00      0.79      0.88        14

    accuracy                           0.91        54
   macro avg       0.94      0.89      0.91        54
weighted avg       0.93      0.91      0.91        54

Confusion Matrix:
[[17  2  0]
 [ 0 21  0]
 [ 0  3 11]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
# Step 8: Convert the trained model to TFLite format
import os

# Workaround: save as SavedModel first, then convert
saved_model_path = "model_saved"
model.export(saved_model_path)

# Convert from SavedModel instead of directly from Keras model
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
tflite_model = converter.convert()

# Save the TFLite model to disk
tflite_path = "model_base.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

# Print file size in kilobytes
file_size_kb = os.path.getsize(tflite_path) / 1024
print(f"TFLite model saved to: '{tflite_path}'")
print(f"File size: {file_size_kb:.2f} KB")

Saved artifact at 'model_saved'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='dense_input')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  136216221221904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136216221223440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136216221222864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136216221224016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136216221223056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136216221222096: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved to: 'model_base.tflite'
File size: 14.02 KB


## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
import os

def file_size_kb(filename):
    return os.path.getsize(filename) / 1024


def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size."""

    # Workaround: save as SavedModel first to avoid Keras/TFLite API mismatch
    saved_model_path = "model_saved_tmp"
    tf.saved_model.save(model, saved_model_path)
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)

    # ------------------------------------------------------------------ #
    # Step 1: Apply quantization settings
    # ------------------------------------------------------------------ #
    if quant_type == 'int8':
        # (a) Enable default optimizations
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Provide representative dataset
        converter.representative_dataset = lambda: representative_data_gen(X_test)
        # (c) Restrict to INT8 ops
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        # (d) Set input/output types to int8
        converter.inference_input_type  = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Set supported types to float16
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations only (weights quantized, activations dynamic)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # ------------------------------------------------------------------ #
    # Step 2: Convert and save
    # ------------------------------------------------------------------ #
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # ------------------------------------------------------------------ #
    # Step 3: Run TFLite inference
    # ------------------------------------------------------------------ #
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_scale,  input_zero_point  = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    is_input_quantized  = input_details["dtype"]  in (np.int8, np.uint8)
    is_output_quantized = output_details["dtype"] in (np.int8, np.uint8)

    y_true = np.argmax(y_test_cat, axis=1)
    y_pred = []

    for i in range(len(X_test)):
        sample = X_test[i:i + 1].astype(np.float32)

        # Quantize input if the model expects int8
        if is_input_quantized:
            sample = (sample / input_scale + input_zero_point).astype(input_details["dtype"])

        interpreter.set_tensor(input_details["index"], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details["index"])

        # Dequantize output if the model returns int8
        if is_output_quantized:
            output = (output.astype(np.float32) - output_zero_point) * output_scale

        y_pred.append(np.argmax(output))

    y_pred = np.array(y_pred)

    # ------------------------------------------------------------------ #
    # Step 4: Report results
    # ------------------------------------------------------------------ #
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")
    print(f"\nClassification Report ({quant_type.upper()}):")
    print(classification_report(y_true, y_pred, target_names=wine.target_names))
    print(f"Confusion Matrix ({quant_type.upper()}):")
    print(confusion_matrix(y_true, y_pred))


# ------------------------------------------------------------------ #
# Run all three quantization modes
# ------------------------------------------------------------------ #
quantize_and_evaluate(model, X_test, y_test_cat, quant_type='dynamic',  filename='model_dynamic.tflite')
quantize_and_evaluate(model, X_test, y_test_cat, quant_type='float16',  filename='model_float16.tflite')
quantize_and_evaluate(model, X_test, y_test_cat, quant_type='int8',     filename='model_int8.tflite')

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



DYNAMIC TFLite model size: 8.44 KB

Classification Report (DYNAMIC):
              precision    recall  f1-score   support

     class_0       1.00      0.89      0.94        19
     class_1       0.81      1.00      0.89        21
     class_2       1.00      0.79      0.88        14

    accuracy                           0.91        54
   macro avg       0.94      0.89      0.91        54
weighted avg       0.93      0.91      0.91        54

Confusion Matrix (DYNAMIC):
[[17  2  0]
 [ 0 21  0]
 [ 0  3 11]]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



FLOAT16 TFLite model size: 8.77 KB

Classification Report (FLOAT16):
              precision    recall  f1-score   support

     class_0       1.00      0.89      0.94        19
     class_1       0.81      1.00      0.89        21
     class_2       1.00      0.79      0.88        14

    accuracy                           0.91        54
   macro avg       0.94      0.89      0.91        54
weighted avg       0.93      0.91      0.91        54

Confusion Matrix (FLOAT16):
[[17  2  0]
 [ 0 21  0]
 [ 0  3 11]]

INT8 TFLite model size: 7.91 KB

Classification Report (INT8):
              precision    recall  f1-score   support

     class_0       0.95      0.95      0.95        19
     class_1       0.75      1.00      0.86        21
     class_2       1.00      0.50      0.67        14

    accuracy                           0.85        54
   macro avg       0.90      0.82      0.82        54
weighted avg       0.88      0.85      0.84        54

Confusion Matrix (INT8):
[[18  1  0]
 [

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
# Step 5: Evaluate all three quantized models
quantize_and_evaluate(model, X_test, y_test_cat, quant_type='dynamic', filename='model_dynamic.tflite')
quantize_and_evaluate(model, X_test, y_test_cat, quant_type='float16', filename='model_float16.tflite')
quantize_and_evaluate(model, X_test, y_test_cat, quant_type='int8',    filename='model_int8.tflite')


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



DYNAMIC TFLite model size: 8.44 KB

Classification Report (DYNAMIC):
              precision    recall  f1-score   support

     class_0       1.00      0.89      0.94        19
     class_1       0.81      1.00      0.89        21
     class_2       1.00      0.79      0.88        14

    accuracy                           0.91        54
   macro avg       0.94      0.89      0.91        54
weighted avg       0.93      0.91      0.91        54

Confusion Matrix (DYNAMIC):
[[17  2  0]
 [ 0 21  0]
 [ 0  3 11]]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



FLOAT16 TFLite model size: 8.77 KB

Classification Report (FLOAT16):
              precision    recall  f1-score   support

     class_0       1.00      0.89      0.94        19
     class_1       0.81      1.00      0.89        21
     class_2       1.00      0.79      0.88        14

    accuracy                           0.91        54
   macro avg       0.94      0.89      0.91        54
weighted avg       0.93      0.91      0.91        54

Confusion Matrix (FLOAT16):
[[17  2  0]
 [ 0 21  0]
 [ 0  3 11]]

INT8 TFLite model size: 7.91 KB

Classification Report (INT8):
              precision    recall  f1-score   support

     class_0       0.95      0.95      0.95        19
     class_1       0.75      1.00      0.86        21
     class_2       1.00      0.50      0.67        14

    accuracy                           0.85        54
   macro avg       0.90      0.82      0.82        54
weighted avg       0.88      0.85      0.84        54

Confusion Matrix (INT8):
[[18  1  0]
 [

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Problem 1 - Part (c)

### Pruning

In [17]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
# Manual pruning implementation (no tfmot required)

# ------------------------------------------------------------------ #
# Step 1: Define PolynomialDecay sparsity schedule
# ------------------------------------------------------------------ #
initial_sparsity = 0.50
final_sparsity   = 0.70
epochs           = 20
batch_size       = 8
dataset_size     = len(X_train)

end_step = int(np.ceil(dataset_size / batch_size)) * epochs
print(f"Dataset size:    {dataset_size}")
print(f"Steps per epoch: {int(np.ceil(dataset_size / batch_size))}")
print(f"Total end_step:  {end_step}")

def polynomial_decay_sparsity(step, initial_sparsity, final_sparsity, end_step, power=3):
    """Mirrors tfmot.sparsity.keras.PolynomialDecay sparsity at a given step."""
    step  = min(step, end_step)
    scale = (1 - step / end_step) ** power
    return final_sparsity + (initial_sparsity - final_sparsity) * scale

def apply_pruning_masks(model, sparsity):
    """Zero out the lowest-magnitude weights to reach the target sparsity."""
    for layer in model.layers:
        if not layer.get_weights():
            continue
        weights = layer.get_weights()
        pruned  = []
        for w in weights:
            if w.ndim < 2:          # skip biases
                pruned.append(w)
                continue
            flat      = np.abs(w.flatten())
            threshold = np.percentile(flat, sparsity * 100)
            mask      = np.abs(w) >= threshold
            pruned.append(w * mask)
        layer.set_weights(pruned)

def count_sparsity(model):
    """Return the fraction of zero weights across all layers."""
    total = zeros = 0
    for layer in model.layers:
        for w in layer.get_weights():
            if w.ndim < 2:
                continue
            total += w.size
            zeros += np.sum(w == 0)
    return zeros / total if total > 0 else 0.0

# ------------------------------------------------------------------ #
# Step 2: Build a fresh model and compile it
# ------------------------------------------------------------------ #
pruned_model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ------------------------------------------------------------------ #
# Step 3: Train with pruning applied at the end of each epoch
# ------------------------------------------------------------------ #
steps_per_epoch = int(np.ceil(dataset_size / batch_size))
pruning_history = {"sparsity": [], "val_accuracy": []}

for epoch in range(epochs):
    hist = pruned_model.fit(
        X_train, y_train_cat,
        epochs          = 1,
        batch_size      = batch_size,
        validation_split= 0.2,
        verbose         = 0
    )

    # Compute sparsity for this epoch's final step
    current_step     = (epoch + 1) * steps_per_epoch
    target_sparsity  = polynomial_decay_sparsity(
                           current_step, initial_sparsity, final_sparsity, end_step)
    apply_pruning_masks(pruned_model, target_sparsity)
    actual_sparsity  = count_sparsity(pruned_model)

    val_acc = hist.history["val_accuracy"][0]
    pruning_history["sparsity"].append(actual_sparsity)
    pruning_history["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch+1:02d}/{epochs} — "
          f"val_accuracy: {val_acc:.4f} — "
          f"sparsity: {actual_sparsity:.4f}")

print(f"\nFinal sparsity: {count_sparsity(pruned_model):.4f}")

Dataset size:    124
Steps per epoch: 16
Total end_step:  320
Epoch 01/20 — val_accuracy: 0.4400 — sparsity: 0.5286
Epoch 02/20 — val_accuracy: 0.2800 — sparsity: 0.5541
Epoch 03/20 — val_accuracy: 0.2800 — sparsity: 0.5769
Epoch 04/20 — val_accuracy: 0.2800 — sparsity: 0.5974
Epoch 05/20 — val_accuracy: 0.5600 — sparsity: 0.6156
Epoch 06/20 — val_accuracy: 0.1200 — sparsity: 0.6310
Epoch 07/20 — val_accuracy: 0.4800 — sparsity: 0.6452
Epoch 08/20 — val_accuracy: 0.4800 — sparsity: 0.6566
Epoch 09/20 — val_accuracy: 0.3200 — sparsity: 0.6667
Epoch 10/20 — val_accuracy: 0.4800 — sparsity: 0.6747
Epoch 11/20 — val_accuracy: 0.6400 — sparsity: 0.6815
Epoch 12/20 — val_accuracy: 0.4800 — sparsity: 0.6872
Epoch 13/20 — val_accuracy: 0.5600 — sparsity: 0.6912
Epoch 14/20 — val_accuracy: 0.6400 — sparsity: 0.6942
Epoch 15/20 — val_accuracy: 0.6000 — sparsity: 0.6969
Epoch 16/20 — val_accuracy: 0.6000 — sparsity: 0.6983
Epoch 17/20 — val_accuracy: 0.6000 — sparsity: 0.6993
Epoch 18/20 — val_ac

In [18]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
# Step 2: Build a prunable Sequential model
# Note: without tfmot, pruning is applied manually via apply_pruning_masks()
# instead of wrapping layers with prune_low_magnitude()

pruned_model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

pruned_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 64)                896       
                                                                 
 dense_7 (Dense)             (None, 32)                2080      
                                                                 
 dense_8 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [19]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
# Step 3: Compile and train the pruned model with manual pruning callbacks

epochs     = 10
batch_size = 8

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Manual equivalent of UpdatePruningStep callback
steps_per_epoch = int(np.ceil(len(X_train) / batch_size))
pruning_history = {"sparsity": [], "val_accuracy": [], "accuracy": []}

for epoch in range(epochs):
    hist = pruned_model.fit(
        X_train, y_train_cat,
        epochs           = 1,
        batch_size       = batch_size,
        validation_split = 0.2,
        verbose          = 0
    )

    # Update pruning masks using PolynomialDecay schedule
    current_step    = (epoch + 1) * steps_per_epoch
    target_sparsity = polynomial_decay_sparsity(
                          current_step, initial_sparsity, final_sparsity, end_step)
    apply_pruning_masks(pruned_model, target_sparsity)
    actual_sparsity = count_sparsity(pruned_model)

    acc     = hist.history["accuracy"][0]
    val_acc = hist.history["val_accuracy"][0]
    pruning_history["accuracy"].append(acc)
    pruning_history["val_accuracy"].append(val_acc)
    pruning_history["sparsity"].append(actual_sparsity)

    print(f"Epoch {epoch+1:02d}/{epochs} — "
          f"accuracy: {acc:.4f} — "
          f"val_accuracy: {val_acc:.4f} — "
          f"sparsity: {actual_sparsity:.4f}")

print(f"\nFinal sparsity : {count_sparsity(pruned_model):.4f}")
print(f"Final val_acc  : {pruning_history['val_accuracy'][-1]:.4f}")

Epoch 01/10 — accuracy: 0.3737 — val_accuracy: 0.1600 — sparsity: 0.5286
Epoch 02/10 — accuracy: 0.2525 — val_accuracy: 0.2800 — sparsity: 0.5541
Epoch 03/10 — accuracy: 0.3939 — val_accuracy: 0.2800 — sparsity: 0.5769
Epoch 04/10 — accuracy: 0.2828 — val_accuracy: 0.4400 — sparsity: 0.5974
Epoch 05/10 — accuracy: 0.6263 — val_accuracy: 0.6000 — sparsity: 0.6156
Epoch 06/10 — accuracy: 0.4646 — val_accuracy: 0.4800 — sparsity: 0.6310
Epoch 07/10 — accuracy: 0.5859 — val_accuracy: 0.3200 — sparsity: 0.6452
Epoch 08/10 — accuracy: 0.4949 — val_accuracy: 0.6000 — sparsity: 0.6566
Epoch 09/10 — accuracy: 0.3232 — val_accuracy: 0.4800 — sparsity: 0.6667
Epoch 10/10 — accuracy: 0.5253 — val_accuracy: 0.4000 — sparsity: 0.6747

Final sparsity : 0.6747
Final val_acc  : 0.4000


In [20]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
# Step 4: Convert pruned model to TFLite and save as "model_pruned.tflite"
# Note: strip_pruning() is not needed since we used manual pruning (no wrapper variables added)

saved_model_path  = "pruned_model_saved"
pruned_tflite_path = "model_pruned.tflite"

# Save as SavedModel first (workaround for TFLite converter bug)
tf.saved_model.save(pruned_model, saved_model_path)

# Convert to TFLite
converter    = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
tflite_model = converter.convert()

# Save to disk
with open(pruned_tflite_path, "wb") as f:
    f.write(tflite_model)

# Print file size
pruned_size_kb = file_size_kb(pruned_tflite_path)
base_size_kb   = file_size_kb("model_base.tflite")

print(f"Pruned TFLite model saved to: '{pruned_tflite_path}'")
print(f"Pruned model size : {pruned_size_kb:.2f} KB")
print(f"Baseline model size: {base_size_kb:.2f} KB")
print(f"Size reduction    : {(1 - pruned_size_kb / base_size_kb) * 100:.1f}%")
print(f"Final sparsity    : {count_sparsity(pruned_model):.4f}")


Pruned TFLite model saved to: 'model_pruned.tflite'
Pruned model size : 14.05 KB
Baseline model size: 14.02 KB
Size reduction    : -0.2%
Final sparsity    : 0.6747


In [21]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
# Step 5: Evaluate the pruned TFLite model on test data

interpreter = tf.lite.Interpreter(model_path="model_pruned.tflite")
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_scale,  input_zero_point  = input_details["quantization"]
output_scale, output_zero_point = output_details["quantization"]

is_input_quantized  = input_details["dtype"]  in (np.int8, np.uint8)
is_output_quantized = output_details["dtype"] in (np.int8, np.uint8)

y_true = np.argmax(y_test_cat, axis=1)
y_pred = []

for i in range(len(X_test)):
    sample = X_test[i:i + 1].astype(np.float32)

    # Quantize input if needed
    if is_input_quantized:
        sample = (sample / input_scale + input_zero_point).astype(input_details["dtype"])

    interpreter.set_tensor(input_details["index"], sample)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])

    # Dequantize output if needed
    if is_output_quantized:
        output = (output.astype(np.float32) - output_zero_point) * output_scale

    y_pred.append(np.argmax(output))

y_pred = np.array(y_pred)

# Results
print(f"Pruned model size : {file_size_kb('model_pruned.tflite'):.2f} KB")
print(f"Final sparsity    : {count_sparsity(pruned_model):.4f}")

print("\nClassification Report (Pruned):")
print(classification_report(y_true, y_pred, target_names=wine.target_names))

print("Confusion Matrix (Pruned):")
print(confusion_matrix(y_true, y_pred))

Pruned model size : 14.05 KB
Final sparsity    : 0.6747

Classification Report (Pruned):
              precision    recall  f1-score   support

     class_0       1.00      0.16      0.27        19
     class_1       0.41      1.00      0.58        21
     class_2       0.00      0.00      0.00        14

    accuracy                           0.44        54
   macro avg       0.47      0.39      0.29        54
weighted avg       0.51      0.44      0.32        54

Confusion Matrix (Pruned):
[[ 3 16  0]
 [ 0 21  0]
 [ 0 14  0]]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

## Problem 1 - Part (d)

### Knowledge Distillation

In [22]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
# Step 1: Define the Student model (smaller architecture)
student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 32)                448       
                                                                 
 dense_10 (Dense)            (None, 16)                528       
                                                                 
 dense_11 (Dense)            (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [23]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
# Step 2: Generate teacher soft labels on training data
teacher_predictions = model.predict(X_train, verbose=0)

print("Teacher predictions shape:", teacher_predictions.shape)
print("\nSample soft labels (first 5 rows):")
print(np.round(teacher_predictions[:5], 4))
print("\nSample hard labels (first 5):")
print(np.argmax(teacher_predictions[:5], axis=1))
print("\nGround truth labels (first 5):")
print(y_train[:5])

Teacher predictions shape: (124, 3)

Sample soft labels (first 5 rows):
[[2.300e-03 3.511e-01 6.466e-01]
 [7.940e-02 7.375e-01 1.831e-01]
 [1.000e-04 9.551e-01 4.480e-02]
 [2.671e-01 5.173e-01 2.156e-01]
 [0.000e+00 9.750e-01 2.500e-02]]

Sample hard labels (first 5):
[2 1 1 1 1]

Ground truth labels (first 5):
[2 1 1 0 1]


In [24]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Step 3: Combine labels and define distillation loss

# (a) Concatenate hard and soft labels along axis=1
y_train_combined = np.concatenate([y_train_cat, teacher_predictions], axis=1)

print("y_train_cat shape      :", y_train_cat.shape)
print("teacher_predictions shape:", teacher_predictions.shape)
print("y_train_combined shape :", y_train_combined.shape)
print("\nSample combined label (first row):")
print(np.round(y_train_combined[0], 4))
print("  └─ hard labels [:3] :", np.round(y_train_combined[0, :3], 4))
print("  └─ soft labels [3:] :", np.round(y_train_combined[0, 3:], 4))

# (b) Define the distillation loss
alpha = 0.5  # weight between hard and soft loss

def distillation_loss(y_true_combined, y_pred):
    """
    Weighted combination of hard and soft label losses.

    Parameters
    ----------
    y_true_combined : tensor, shape (batch, 6)
        First 3 columns  → one-hot hard labels
        Last  3 columns  → teacher soft label probabilities
    y_pred : tensor, shape (batch, 3)
        Student model softmax predictions
    alpha : float
        Weight for hard loss; (1 - alpha) applied to soft loss
    """
    # Split combined labels back into hard and soft
    y_true_hard = y_true_combined[:, :num_classes]      # shape (batch, 3)
    y_true_soft = y_true_combined[:, num_classes:]      # shape (batch, 3)

    # Hard loss: student predictions vs ground truth one-hot labels
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)

    # Soft loss: student predictions vs teacher soft probabilities
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    # Weighted combination
    return alpha * hard_loss + (1 - alpha) * soft_loss


# Quick sanity check
sample_pred     = teacher_predictions[:4]               # use teacher preds as dummy student preds
sample_combined = y_train_combined[:4]
sample_loss     = distillation_loss(
                    tf.constant(sample_combined, dtype=tf.float32),
                    tf.constant(sample_pred,     dtype=tf.float32))

print("\nSanity check — loss on 4 samples:")
print(np.round(sample_loss.numpy(), 6))

y_train_cat shape      : (124, 3)
teacher_predictions shape: (124, 3)
y_train_combined shape : (124, 6)

Sample combined label (first row):
[0.     0.     1.     0.0023 0.3511 0.6466]
  └─ hard labels [:3] : [0. 0. 1.]
  └─ soft labels [3:] : [0.0023 0.3511 0.6466]

Sanity check — loss on 4 samples:
[0.549588 0.520485 0.114848 1.172321]


In [25]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
# Step 4: Compile and train the student model with distillation loss
student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

# Train using combined labels (hard + soft)
student_history = student_model.fit(
    X_train, y_train_combined,
    epochs           = 10,
    batch_size       = 8,
    validation_split = 0.2,
    verbose          = 1
)

Epoch 1/10
13/13 [==============================] - 4s 33ms/step - loss: 5.1406 - accuracy: 0.3333 - val_loss: 1.2846 - val_accuracy: 0.1600
Epoch 2/10
13/13 [==============================] - 0s 16ms/step - loss: 2.7401 - accuracy: 0.3535 - val_loss: 2.0042 - val_accuracy: 0.6400
Epoch 3/10
13/13 [==============================] - 0s 16ms/step - loss: 1.8298 - accuracy: 0.3636 - val_loss: 1.3659 - val_accuracy: 0.5200
Epoch 4/10
13/13 [==============================] - 0s 13ms/step - loss: 1.2081 - accuracy: 0.3535 - val_loss: 0.9271 - val_accuracy: 0.6000
Epoch 5/10
13/13 [==============================] - 0s 11ms/step - loss: 0.8193 - accuracy: 0.5556 - val_loss: 1.0493 - val_accuracy: 0.6800
Epoch 6/10
13/13 [==============================] - 0s 11ms/step - loss: 0.7503 - accuracy: 0.5960 - val_loss: 0.7597 - val_accuracy: 0.4400
Epoch 7/10
13/13 [==============================] - 0s 10ms/step - loss: 0.6833 - accuracy: 0.6465 - val_loss: 0.9713 - val_accuracy: 0.6400
Epoch 8/10
13

In [26]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
# Step 5: Convert the student model to TFLite and save as "model_kd.tflite"
kd_tflite_path   = "model_kd.tflite"
saved_model_path = "student_model_saved"

# Workaround: save as SavedModel first to avoid TFLite converter bug
tf.saved_model.save(student_model, saved_model_path)

# Convert to TFLite
converter    = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
tflite_model = converter.convert()

# Save to disk
with open(kd_tflite_path, "wb") as f:
    f.write(tflite_model)

# Print and compare file sizes
kd_size_kb   = file_size_kb(kd_tflite_path)
base_size_kb = file_size_kb("model_base.tflite")

print(f"Student (KD) TFLite saved to : '{kd_tflite_path}'")
print(f"Student (KD) model size      : {kd_size_kb:.2f} KB")
print(f"Baseline model size          : {base_size_kb:.2f} KB")
print(f"Size reduction               : {(1 - kd_size_kb / base_size_kb) * 100:.1f}%")

Student (KD) TFLite saved to : 'model_kd.tflite'
Student (KD) model size      : 6.05 KB
Baseline model size          : 14.02 KB
Size reduction               : 56.8%


In [27]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
# Step 6: Evaluate the student (KD) model on test data
y_pred_probs = student_model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = np.argmax(y_test_cat,   axis=1)

print(f"Student (KD) model size: {file_size_kb('model_kd.tflite'):.2f} KB")
print(f"Student parameters     : 1,027  (vs teacher: 3,075)\n")

print("Classification Report (Student — Knowledge Distillation):")
print(classification_report(y_true, y_pred, target_names=wine.target_names))

print("Confusion Matrix (Student — Knowledge Distillation):")
print(confusion_matrix(y_true, y_pred))

Student (KD) model size: 6.05 KB
Student parameters     : 1,027  (vs teacher: 3,075)

Classification Report (Student — Knowledge Distillation):
              precision    recall  f1-score   support

     class_0       1.00      0.79      0.88        19
     class_1       0.86      0.29      0.43        21
     class_2       0.41      0.93      0.57        14

    accuracy                           0.63        54
   macro avg       0.75      0.67      0.63        54
weighted avg       0.79      0.63      0.62        54

Confusion Matrix (Student — Knowledge Distillation):
[[15  0  4]
 [ 0  6 15]
 [ 0  1 13]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [ ]:
# <-- (if needed) Enter your code here <--#

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
